# 🧠 DiabetAI v8 - Physics-Constrained XGBoost

**Problem:** v7 model ignores carbs, predicts ~0 for high carb meals

**Solution:** Monotone constraints + Sanity validation

In [ ]:
!pip install polars xgboost matplotlib -U -q

import polars as pl
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import json
import os

from google.colab import drive
drive.mount('/content/drive')

print(f"✅ XGBoost: {xgb.__version__}")
print(f"✅ Polars: {pl.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION - STRICT FEATURE DEFINITION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
MODEL_PATH = "/content/drive/MyDrive/DiabetesKDS/models"

TRAIN_FILE = f"{BASE_PATH}/processed/train_ready_balanced_v7.parquet"
TEST_FILE = f"{BASE_PATH}/processed/test_ready_real_world_v7.parquet"

# 17 FEATURES - STRICT ORDER (Must match app.py EXACTLY)
FEATURES = [
    'CGM',              # 0
    'carbs',            # 1 <- CRITICAL: Must be positive constrained
    'iob',              # 2
    'est_tdd',          # 3
    'est_isf',          # 4
    'est_icr',          # 5
    'cgm_delta_5min',   # 6
    'cgm_delta_15min',  # 7
    'cgm_velocity',     # 8
    'basal',            # 9
    'basal_delta_5min', # 10
    'time_sin',         # 11
    'time_cos',         # 12
    'is_weekend',       # 13
    'age',              # 14
    'weight',           # 15
    'gender_encoded'    # 16
]

TARGET = 'bolus'

print(f"📋 Features: {len(FEATURES)}")
print(f"🎯 Target: {TARGET}")
print(f"\n✅ 'carbs' at index: {FEATURES.index('carbs')}")

## Step 1: Load Data

In [ ]:
print("="*70)
print("📂 LOADING DATA")
print("="*70)

df_train = pl.read_parquet(TRAIN_FILE)
df_test = pl.read_parquet(TEST_FILE)

print(f"\n✅ Train: {len(df_train):,} rows")
print(f"✅ Test: {len(df_test):,} rows")

# Check features exist
missing = [f for f in FEATURES if f not in df_train.columns]
if missing:
    raise ValueError(f"❌ Missing features in data: {missing}")
print(f"\n✅ All {len(FEATURES)} features found!")

In [ ]:
# Prepare feature matrices
print("\n🔧 Preparing feature matrices...")

X_train = df_train.select(FEATURES).to_numpy().astype(np.float32)
y_train = df_train.select(TARGET).to_numpy().ravel().astype(np.float32)

X_test = df_test.select(FEATURES).to_numpy().astype(np.float32)
y_test = df_test.select(TARGET).to_numpy().ravel().astype(np.float32)

print(f"\n📊 Train: X={X_train.shape}, y={y_train.shape}")
print(f"📊 Test:  X={X_test.shape}, y={y_test.shape}")

# Check for inf/nan
print(f"\n🔍 Train NaN: {np.isnan(X_train).sum()}, Inf: {np.isinf(X_train).sum()}")
print(f"🔍 Test NaN: {np.isnan(X_test).sum()}, Inf: {np.isinf(X_test).sum()}")

## Step 2: Physics-Informed Training (Monotone Constraints)

In [ ]:
print("="*70)
print("🧠 MONOTONE CONSTRAINT SETUP")
print("="*70)

# ============================================================
# MONOTONE CONSTRAINTS - THE PHYSICS LAYER
# ============================================================
# +1 = Positive (as feature increases, prediction must increase or stay same)
# -1 = Negative (as feature increases, prediction must decrease or stay same)
#  0 = No constraint

# Build constraint tuple
monotone = [0] * len(FEATURES)  # Start with no constraints

# CRITICAL: Carbs MUST increase bolus
carbs_idx = FEATURES.index('carbs')
monotone[carbs_idx] = 1  # Positive constraint

# CGM: Higher glucose = more correction bolus (positive)
cgm_idx = FEATURES.index('CGM')
monotone[cgm_idx] = 1

# IOB: Higher IOB = less bolus needed (negative)
iob_idx = FEATURES.index('iob')
monotone[iob_idx] = -1

monotone_tuple = tuple(monotone)

print(f"\n📋 Monotone Constraints:")
for i, (feat, constraint) in enumerate(zip(FEATURES, monotone)):
    if constraint != 0:
        symbol = "⬆️ +1" if constraint == 1 else "⬇️ -1"
        print(f"   {i}: {feat} → {symbol}")

In [ ]:
print("\n" + "="*70)
print("🚀 TRAINING XGBOOST v8")
print("="*70)

# Check GPU
try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode == 0:
        device = 'cuda'
        print("\n🚀 GPU detected - CUDA acceleration ON")
    else:
        device = 'cpu'
except:
    device = 'cpu'
    print("\n💻 Using CPU")

# Model parameters
params = {
    'n_estimators': 2000,
    'learning_rate': 0.02,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': device,
    'random_state': 42,
    'monotone_constraints': monotone_tuple,  # THE PHYSICS LAYER
    'early_stopping_rounds': 50
}

print(f"\n📋 Parameters:")
for k, v in params.items():
    if k != 'monotone_constraints':
        print(f"   {k}: {v}")
print(f"   monotone_constraints: ACTIVE ({sum(1 for x in monotone if x != 0)} features constrained)")

In [ ]:
# Train model
print("\n🏋️ Training...")

model = xgb.XGBRegressor(**params)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

print(f"\n✅ Training complete!")
print(f"   Best iteration: {model.best_iteration}")
print(f"   Best score: {model.best_score:.6f}")

## Step 3: THE GATEKEEPER - Sanity Validation

In [ ]:
print("="*70)
print("🚨 GATEKEEPER VALIDATION - SANITY CHECK")
print("="*70)

# Create a synthetic test case
# Scenario: 150 mg/dL glucose, 60g carbs, no IOB, typical patient
# Expected: With ICR=10, this should be ~6 units (60/10)

sanity_input = {
    'CGM': 150.0,
    'carbs': 60.0,         # 60g meal
    'iob': 0.0,
    'est_tdd': 40.0,
    'est_isf': 30.0,
    'est_icr': 10.0,
    'cgm_delta_5min': 0.0,
    'cgm_delta_15min': 0.0,
    'cgm_velocity': 0.0,
    'basal': 1.0,
    'basal_delta_5min': 0.0,
    'time_sin': 0.0,
    'time_cos': 1.0,
    'is_weekend': 0.0,
    'age': 30.0,
    'weight': 70.0,
    'gender_encoded': 0.0
}

# Build feature array in correct order
sanity_array = np.array([[sanity_input[f] for f in FEATURES]], dtype=np.float32)

print(f"\n🧪 Test Case:")
print(f"   CGM: {sanity_input['CGM']} mg/dL")
print(f"   Carbs: {sanity_input['carbs']}g")
print(f"   IOB: {sanity_input['iob']}U")
print(f"   ICR: {sanity_input['est_icr']}")
print(f"\n   Expected (Math): ~{sanity_input['carbs'] / sanity_input['est_icr']:.1f}U")

# Predict
prediction = model.predict(sanity_array)[0]

print(f"\n🎯 MODEL PREDICTION: {prediction:.2f} Units")

In [ ]:
# THE ASSERTION - CRITICAL GATE
MIN_EXPECTED = 3.0  # For 60g carbs, at least 3U is expected

if prediction < MIN_EXPECTED:
    error_msg = f"""
    ╔══════════════════════════════════════════════════════════════╗
    ║  ❌ FATAL: MODEL IS BLIND TO CARBS!                         ║
    ║                                                              ║
    ║  Predicted: {prediction:.2f}U for 60g carbs                          ║
    ║  Expected:  >= {MIN_EXPECTED:.2f}U                                       ║
    ║                                                              ║
    ║  TRAINING REJECTED - MODEL NOT SAVED                        ║
    ╚══════════════════════════════════════════════════════════════╝
    """
    print(error_msg)
    raise RuntimeError(f"Model is blind to carbs! Predicted {prediction:.2f}U for 60g carbs.")
else:
    print(f"""
    ╔══════════════════════════════════════════════════════════════╗
    ║  ✅ SANITY CHECK PASSED!                                    ║
    ║                                                              ║
    ║  60g Carbs → {prediction:.2f}U (Expected >= {MIN_EXPECTED:.2f}U)                  ║
    ║                                                              ║
    ║  Proceeding to save model...                                ║
    ╚══════════════════════════════════════════════════════════════╝
    """)

In [ ]:
# Additional sanity checks
print("\n📊 ADDITIONAL CARB RESPONSE TESTS:")
print("-" * 50)

for carb_test in [0, 30, 60, 90, 120, 150]:
    test_input = sanity_input.copy()
    test_input['carbs'] = float(carb_test)
    test_array = np.array([[test_input[f] for f in FEATURES]], dtype=np.float32)
    pred = model.predict(test_array)[0]
    print(f"   {carb_test:3d}g Carbs → {pred:.2f}U")

print("\n🔍 Verifying monotone constraint (carbs should increase dose)...")

## Step 4: Save Model (Only if Passed)

In [ ]:
print("="*70)
print("💾 SAVING MODEL")
print("="*70)

os.makedirs(MODEL_PATH, exist_ok=True)

# Save model
model_file = f"{MODEL_PATH}/xgboost_clinical_v8.json"
model.save_model(model_file)
print(f"\n✅ Model saved: {model_file}")

# Save feature list (for app.py sync)
features_file = f"{MODEL_PATH}/model_features_v8.json"
with open(features_file, 'w') as f:
    json.dump({
        'features': FEATURES,
        'target': TARGET,
        'monotone_constraints': list(monotone_tuple),
        'version': 'v8',
        'sanity_check': {
            'test_carbs': 60,
            'prediction': float(prediction)
        }
    }, f, indent=2)
print(f"✅ Features saved: {features_file}")

## Step 5: Visualization & Verification

In [ ]:
print("="*70)
print("📊 FEATURE IMPORTANCE")
print("="*70)

importance = model.feature_importances_
sorted_idx = np.argsort(importance)[::-1]

# Check carbs is in top 3
carbs_rank = np.where(sorted_idx == carbs_idx)[0][0] + 1
print(f"\n🥇 Carbs Importance Rank: #{carbs_rank}")

if carbs_rank > 5:
    print(f"⚠️ WARNING: Carbs ranked #{carbs_rank}, expected in top 3!")

# Plot
plt.figure(figsize=(12, 6))
top_n = 10
top_features = [FEATURES[i] for i in sorted_idx[:top_n]]
top_importance = importance[sorted_idx[:top_n]]

colors = ['#ff4b4b' if f == 'carbs' else '#667eea' for f in top_features]

plt.barh(range(top_n), top_importance[::-1], color=colors[::-1])
plt.yticks(range(top_n), top_features[::-1])
plt.xlabel('Importance')
plt.title('v8 Feature Importance (Carbs in RED)')
plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/v8_feature_importance.png", dpi=150)
plt.show()

In [ ]:
print("="*70)
print("📊 CARBS vs PREDICTED BOLUS (Monotone Verification)")
print("="*70)

# Get test predictions
y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, 0, 25)  # Safety clip

# Get carbs column from test
test_carbs = X_test[:, carbs_idx]

# Sample for plotting (too many points = slow)
sample_size = min(10000, len(test_carbs))
indices = np.random.choice(len(test_carbs), sample_size, replace=False)

plt.figure(figsize=(10, 6))
plt.scatter(test_carbs[indices], y_pred[indices], alpha=0.3, s=5)
plt.xlabel('Carbs (g)')
plt.ylabel('Predicted Bolus (U)')
plt.title('v8 Model: Carbs vs Predicted Bolus\n(Should show POSITIVE correlation)')

# Add trend line
z = np.polyfit(test_carbs[indices], y_pred[indices], 1)
p = np.poly1d(z)
x_line = np.linspace(0, test_carbs.max(), 100)
plt.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Trend (slope={z[0]:.3f})')
plt.legend()

plt.xlim(0, 200)
plt.ylim(0, 15)
plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/v8_carbs_vs_bolus.png", dpi=150)
plt.show()

# Verify positive correlation
if z[0] > 0:
    print(f"\n✅ Positive correlation confirmed! Slope = {z[0]:.4f}")
else:
    print(f"\n❌ WARNING: Negative slope detected! Slope = {z[0]:.4f}")

In [ ]:
# Final Summary
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*70)
print("🎉 v8 TRAINING SUMMARY")
print("="*70)
print(f"\n📊 Metrics:")
print(f"   RMSE: {rmse:.4f}")
print(f"   MAE:  {mae:.4f}")
print(f"\n📋 Model:")
print(f"   Features: {len(FEATURES)}")
print(f"   Constrained: carbs(+1), CGM(+1), IOB(-1)")
print(f"   Sanity: 60g→{prediction:.2f}U ✅")
print(f"\n💾 Files:")
print(f"   {model_file}")
print(f"   {features_file}")
print("\n🚀 Ready for deployment!")